# 01 — Exploratory Data Analysis

**Project**: Auto Tagging Ticket Support System  
**Dataset**: Customer Support Ticket Dataset (Kaggle)  

## Goals
- Understand class distribution (label balance)
- Analyze text length characteristics
- Generate word clouds per category
- Identify class imbalance issues
- Sample data per category

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from wordcloud import WordCloud
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 200)
print('Libraries loaded successfully.')

## 1. Load Data

In [ ]:
from src.preprocessing import load_and_prepare_data

DATA_PATH = '../data/raw/customer_support_tickets.csv'

df = load_and_prepare_data(DATA_PATH)
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
print('=== Dataset Info ===')
print(f'Total rows      : {len(df):,}')
print(f'Unique labels   : {df["label"].nunique()}')
print(f'Missing text    : {df["text"].isna().sum()}')
print(f'Missing label   : {df["label"].isna().sum()}')
print()
print('Label counts:')
print(df['label'].value_counts())

## 2. Class Distribution

In [ ]:
label_counts = df['label'].value_counts().reset_index()
label_counts.columns = ['Category', 'Count']
label_counts['Percentage'] = (label_counts['Count'] / len(df) * 100).round(2)

fig = px.bar(
    label_counts,
    x='Count', y='Category',
    orientation='h',
    color='Count',
    color_continuous_scale='Blues',
    title='Ticket Count per Category',
    text='Count'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, height=400)
fig.show()

label_counts

In [ ]:
# Class imbalance check
max_count = label_counts['Count'].max()
min_count = label_counts['Count'].min()
imbalance_ratio = max_count / min_count

print(f'Max class count  : {max_count:,}')
print(f'Min class count  : {min_count:,}')
print(f'Imbalance ratio  : {imbalance_ratio:.2f}x')

if imbalance_ratio > 5:
    print('⚠  High class imbalance detected! Consider oversampling or class_weight="balanced".')
elif imbalance_ratio > 2:
    print('⚠  Moderate class imbalance. Monitor per-class F1 scores carefully.')
else:
    print('✓  Dataset is reasonably balanced.')

## 3. Text Length Analysis

In [ ]:
df['char_count'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print('=== Text Length Statistics ===')
print(df[['char_count', 'word_count']].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['char_count'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Character Count')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['word_count'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Distribution of Word Count')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Word count per category (box plot)
fig = px.box(
    df, x='label', y='word_count',
    color='label',
    title='Word Count Distribution per Category',
    labels={'label': 'Category', 'word_count': 'Word Count'}
)
fig.update_layout(showlegend=False)
fig.show()

## 4. Word Cloud per Category

In [ ]:
from src.preprocessing import preprocess_series

print('Preprocessing text (this may take a moment)...')
df['text_clean'] = preprocess_series(df['text'])
print('Done.')

In [ ]:
categories = sorted(df['label'].unique())
n_cats = len(categories)
cols = 3
rows = (n_cats + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 5))
axes = axes.flatten()

for i, cat in enumerate(categories):
    corpus = ' '.join(df[df['label'] == cat]['text_clean'].fillna(''))
    if corpus.strip():
        wc = WordCloud(width=500, height=280, background_color='white',
                       colormap='tab10', max_words=80).generate(corpus)
        axes[i].imshow(wc, interpolation='bilinear')
    axes[i].set_title(cat, fontsize=12, fontweight='bold')
    axes[i].axis('off')

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Word Clouds per Category', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Top Keywords per Category

In [ ]:
TOP_N = 15
n_cats = len(categories)
cols = 2
rows = (n_cats + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 5))
axes = axes.flatten()

for i, cat in enumerate(categories):
    corpus = ' '.join(df[df['label'] == cat]['text_clean'].fillna(''))
    word_freq = Counter(corpus.split()).most_common(TOP_N)
    if word_freq:
        words, freqs = zip(*word_freq)
        axes[i].barh(list(words)[::-1], list(freqs)[::-1], color='steelblue')
        axes[i].set_title(f'{cat}', fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Frequency')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'Top {TOP_N} Keywords per Category', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Sample Data per Category

In [ ]:
SAMPLES_PER_CAT = 2

for cat in categories:
    print(f'\n{'='*60}')
    print(f'Category: {cat}')
    print('='*60)
    samples = df[df['label'] == cat]['text'].sample(min(SAMPLES_PER_CAT, len(df[df['label'] == cat])), random_state=42)
    for j, text in enumerate(samples, 1):
        print(f'  [{j}] {text[:300]}...' if len(text) > 300 else f'  [{j}] {text}')

## 7. EDA Summary

Fill in after running all cells above:

| Finding | Value |
|---------|-------|
| Total tickets | — |
| Number of categories | — |
| Most common category | — |
| Least common category | — |
| Imbalance ratio | — |
| Avg word count | — |
| Avg char count | — |

**Key Observations**:
1. *(Fill after running)*
2. *(Fill after running)*
3. *(Fill after running)*

In [ ]:
# Save processed data for use in model training notebook
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/tickets_clean.csv', index=False)
print('Saved processed data to data/processed/tickets_clean.csv')